In [3]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001-1028f23e353fbe3e.parquet', 'validation': 'data/validation-00000-of-00001-6c7328ff6c84284c.parquet', 'test': 'data/test-00000-of-00001-f0e719df791966ff.parquet'}
df = pd.read_parquet("hf://datasets/derek-thomas/ScienceQA/" + splits["train"])

In [4]:
df.head()

,image,question,choices,answer,hint,task,grade,subject,topic,category,skill,lecture,solution
0,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,Which of these states is farthest north?,"[West Virginia, Louisiana, Arizona, Oklahoma]",0,,closed choice,grade2,social science,geography,Geography,Read a map: cardinal directions,"Maps have four cardinal directions, or main di...","To find the answer, look at the compass rose. ..."
1,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,Identify the question that Tom and Justin's ex...,[Do ping pong balls stop rolling along the gro...,1,The passage below describes an experiment. Rea...,closed choice,grade8,natural science,science-and-engineering-practices,Designing experiments,Identify the experimental question,Experiments can be designed to answer specific...,
2,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,Identify the question that Kathleen and Bryant...,[Does Kathleen's snowboard slide down a hill i...,0,The passage below describes an experiment. Rea...,closed choice,grade7,natural science,science-and-engineering-practices,Designing experiments,Identify the experimental question,Experiments can be designed to answer specific...,
3,None,Which tense does the sentence use?\nMona will ...,"[present tense, future tense, past tense]",1,,closed choice,grade2,language science,verbs,Verb tense,"Is the sentence in the past, present, or futur...",Present tense verbs tell you about something t...,The sentence is in future tense. You can tell ...
4,None,Complete the sentence.\nSewing an apron is a ().,"[chemical change, physical change]",1,,closed choice,grade4,natural science,chemistry,Physical and chemical change,Identify physical and chemical changes,Chemical changes and physical changes are two ...,Sewing an apron is a physical change. The fabr...


# Cleaning

In [7]:
text_columns = ["question", "hint", "lecture", "solution"]

# Function to clean and preprocess text (lowercase, remove punctuation, etc.)
import re

def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text)     # Remove excess whitespace
    return text.strip()

# Apply cleaning
for col in text_columns:
    df[col] = df[col].astype(str).apply(clean_text)


# Combined Text

In [10]:
df["combined_text"] = df[text_columns].agg(' '.join, axis=1)
texts = df["combined_text"].tolist()


# Vectorization from Scratch: Bag-of-Words

In [11]:
from collections import Counter

all_tokens = [token for text in texts for token in text.split()]
vocab = sorted(set(all_tokens))
word_to_index = {word: i for i, word in enumerate(vocab)}


In [19]:
import numpy as np

def vectorize(text, mapping):
    vec = np.zeros(len(mapping), dtype=int)
    for token in text.split():
        if token in mapping:
            vec[mapping[token]] += 1
    return vec

vectors = np.array([vectorize(text, word_to_index) for text in texts])
# vectors.shape = (number_of_samples, vocabulary_size)


In [20]:
print(vectors[3])

[0 0 0 ... 0 0 0]


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_vectors = vectorizer.fit_transform(texts)
# tfidf_vectors is a sparse matrix for efficient storage


In [21]:
from collections import defaultdict
import numpy as np

class SimpleTfidfVectorizer:
    def __init__(self):
        self.vocab = {}
        self.idf = None

    def fit(self, documents):
        # Build vocabulary
        vocab_set = set()
        doc_freq = defaultdict(int)
        for doc in documents:
            words = set(doc.split())
            vocab_set.update(words)
            for word in words:
                doc_freq[word] += 1
        self.vocab = {word: idx for idx, word in enumerate(sorted(vocab_set))}
        N = len(documents)
        # Compute IDF
        self.idf = np.log((1 + N) / (1 + np.array([doc_freq[word] for word in sorted(vocab_set)]))) + 1

    def transform(self, documents):
        tfidf_matrix = np.zeros((len(documents), len(self.vocab)))
        for i, doc in enumerate(documents):
            word_counts = defaultdict(int)
            words = doc.split()
            for word in words:
                if word in self.vocab:
                    word_counts[word] += 1
            for word, count in word_counts.items():
                idx = self.vocab[word]
                tf = count / len(words)
                tfidf_matrix[i, idx] = tf * self.idf[idx]
        return tfidf_matrix

    def fit_transform(self, documents):
        self.fit(documents)
        return self.transform(documents)

# Example usage:
simple_tfidf = SimpleTfidfVectorizer()
rag_tfidf_vectors = simple_tfidf.fit_transform(texts)